# 🌾 AgriHub: Fine-Tuning Multi-Crop ConvNeXt on PlantDoc (Real Field) + PlantVillage (Lab)
### Directly streams datasets from Kaggle via `kagglehub` — No local computer storage or bandwidth used!
This notebook fine-tunes the **ConvNeXt-Small Crop Detector** from Hugging Face (`ixrbhii/multicrop-disease-models`) so that wide-angle field photos (such as Maka / Corn fields in ambient sunlight) are accurately identified as **Maize** instead of being misclassified as Rice!

* **Runtime:** Select `Runtime` -> `Change runtime type` -> `T4 GPU` (Free)
* **Training Time:** ~10-12 minutes for 10 epochs
* **Output:** `model.safetensors` & `config.json` ready for 1-click download directly into AgriHub!


In [ ]:
# Check GPU availability
!nvidia-smi


In [ ]:
# Install required packages
!pip install -q kagglehub timm safetensors huggingface_hub torch torchvision pillow matplotlib


## 📥 Step 1: Stream Datasets Directly From Kaggle
Using `kagglehub`, both datasets download directly into Colab's cloud filesystem in ~60 seconds without using your home internet bandwidth or local disk!


In [ ]:
import kagglehub

print("📥 1. Streaming PlantDoc Dataset from Kaggle (Real in-the-wild field photos)...")
plantdoc_path = kagglehub.dataset_download("abdulhasibuddin/plant-doc-dataset")
print("✓ PlantDoc Path:", plantdoc_path)

print("\n📥 2. Streaming PlantVillage Dataset from Kaggle (Clean leaf studio photos)...")
plantvillage_path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("✓ PlantVillage Path:", plantvillage_path)


## 🏷️ Step 2: Define Taxonomy & Class Mapping
Maps irregular folder names from PlantDoc & PlantVillage to the 36 standard canonical classes expected by AgriHub's ConvNeXt detector.


In [ ]:
import os
import glob
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from safetensors.torch import load_file, save_file
import timm
from huggingface_hub import hf_hub_download
import json
import time
import random
from collections import Counter

# Canonical 36 crop classes matching ixrbhii/multicrop-disease-models detectors/new
CANONICAL_CLASSES = [
    'Apple', 'Apricot', 'Betel', 'Bitter_gourd', 'Black_Pepper', 
    'Bottle_gourd', 'Cashew', 'Cassava', 'Cherry', 'Chrysanthemum', 
    'Citrus', 'Coffee', 'Cucumber', 'Cucurbit', 'Fig', 'Grape', 
    'Loquat', 'Maize', 'Mango', 'Peach', 'Pear', 'Pepper', 'Potato', 
    'Raspberry', 'Rice', 'Ridge_gourd', 'Rose', 'Snake_gourd', 
    'Sorghum', 'Soybean', 'Squash', 'Strawberry', 'Sugarcane', 
    'Tea', 'Tomato', 'Walnut'
]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CANONICAL_CLASSES)}

CROP_ALIAS_MAP = {
    "maize": "Maize", "corn": "Maize", "maka": "Maize", "makka": "Maize", "zea mays": "Maize",
    "rice": "Rice", "paddy": "Rice", "bhat": "Rice", "oryza sativa": "Rice",
    "soybean": "Soybean", "soy": "Soybean",
    "tomato": "Tomato", "potato": "Potato", "grape": "Grape", "grapes": "Grape",
    "citrus": "Citrus", "orange": "Citrus", "lemon": "Citrus", "lime": "Citrus",
    "pepper": "Pepper", "bell pepper": "Pepper", "apple": "Apple",
    "cherry": "Cherry", "peach": "Peach", "strawberry": "Strawberry",
    "squash": "Squash", "sugarcane": "Sugarcane", "mango": "Mango", "raspberry": "Raspberry"
}

def map_folder_to_crop(folder_name: str):
    clean = folder_name.lower().replace("_", " ").replace("-", " ")
    for alias, canonical in CROP_ALIAS_MAP.items():
        if alias in clean:
            return canonical
    return None

print(f"✓ Taxonomy initialized with {len(CANONICAL_CLASSES)} classes.")


## 🔍 Step 3: Discover & Balance Field (PlantDoc) and Lab (PlantVillage) Images


In [ ]:
def scan_datasets(doc_dir, village_dir, max_lab_per_class=600):
    samples = []
    
    # 1. PlantDoc (In-the-wild field photos)
    for p in Path(doc_dir).rglob("*"):
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
            crop = map_folder_to_crop(p.parent.name)
            if not crop and p.parent.parent:
                crop = map_folder_to_crop(p.parent.parent.name)
            if crop and crop in CLASS_TO_IDX:
                samples.append((str(p), CLASS_TO_IDX[crop], "PlantDoc_Field"))
                
    # 2. PlantVillage (Studio leaf photos, capped to prevent dominating field photos)
    lab_counts = {}
    v_root = Path(village_dir) / "color" if (Path(village_dir) / "color").exists() else Path(village_dir)
    for p in v_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
            crop = map_folder_to_crop(p.parent.name)
            if crop and crop in CLASS_TO_IDX:
                cnt = lab_counts.get(crop, 0)
                if cnt < max_lab_per_class:
                    samples.append((str(p), CLASS_TO_IDX[crop], "PlantVillage_Lab"))
                    lab_counts[crop] = cnt + 1
                    
    return samples

all_samples = scan_datasets(plantdoc_path, plantvillage_path)
print(f"✓ Total combined dataset: {len(all_samples)} images!")

dist = Counter([CANONICAL_CLASSES[s[1]] for s in all_samples])
print("\nTop 10 classes in training pool:")
for crop_name, count in sorted(dist.items(), key=lambda x: -x[1])[:10]:
    print(f"  🌿 {crop_name:<16}: {count} images")


## 🎨 Step 4: Data Augmentation & DataLoaders
Includes RandomResizedCrop and ColorJitter to simulate direct sunlight, canopy shading, and varied distances in open farm fields.


In [ ]:
class PlantDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, label, _ = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), (0, 0, 0))
        if self.transform:
            img = self.transform(img)
        return img, label

# 85% Train, 15% Validation split
random.seed(42)
random.shuffle(all_samples)
split_idx = int(len(all_samples) * 0.85)
train_samples = all_samples[:split_idx]
val_samples = all_samples[split_idx:]

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.25),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(PlantDataset(train_samples, train_tf), batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(PlantDataset(val_samples, val_tf), batch_size=16, shuffle=False, num_workers=2)

print(f"✓ DataLoaders ready: {len(train_samples)} Train images, {len(val_samples)} Validation images.")


## 🧠 Step 5: Load Pretrained ConvNeXt Detector from Hugging Face
Loads `ixrbhii/multicrop-disease-models` (`detectors/new`) weights and freezes lower stages to preserve fine pattern recognition.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Device: {device}")

model = timm.create_model("convnext_small", pretrained=False, num_classes=len(CANONICAL_CLASSES))
weights_path = hf_hub_download(repo_id="ixrbhii/multicrop-disease-models", filename="detectors/new/model.safetensors")
state_dict = load_file(weights_path)
model.load_state_dict(state_dict)
model.to(device)

# Freeze stages 0 and 1 for speed and stability
for name, param in model.named_parameters():
    if "stages.0." in name or "stages.1." in name or "stem." in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✓ Pretrained ConvNeXt loaded! Trainable parameters: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")


## 🚀 Step 6: Execute Fine-Tuning Loop
Uses AdamW with Cosine Annealing learning rate schedule and FP16 automatic mixed precision.


In [ ]:
EPOCHS = 10
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

out_dir = Path("./agrihub_finetuned_detector")
out_dir.mkdir(exist_ok=True)
best_acc = 0.0

print(f"Starting fine-tuning for {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tot_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                outs = model(imgs)
                loss = criterion(outs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outs = model(imgs)
            loss = criterion(outs, labels)
            loss.backward()
            optimizer.step()
        tot_loss += loss.item() * imgs.size(0)
        correct += (outs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
        
    scheduler.step()
    train_loss = tot_loss / total
    train_acc = (correct / total) * 100.0
    
    # Validation
    model.eval()
    v_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if scaler:
                with torch.amp.autocast('cuda'):
                    outs = model(imgs)
                    loss = criterion(outs, labels)
            else:
                outs = model(imgs)
                loss = criterion(outs, labels)
            v_loss += loss.item() * imgs.size(0)
            v_corr += (outs.argmax(dim=1) == labels).sum().item()
            v_tot += labels.size(0)
            
    val_loss = v_loss / v_tot
    val_acc = (v_corr / v_tot) * 100.0
    elapsed = time.time() - t0
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")
    
    if val_acc > best_acc:
        best_acc = val_acc
        save_file(model.state_dict(), str(out_dir / "model.safetensors"))
        print(f"  ⭐ Best weights saved! (Val Acc: {best_acc:.2f}%)")

# Save config.json
with open(out_dir / "config.json", "w") as f:
    json.dump({
        "model_architecture": "convnext_small",
        "classes": CANONICAL_CLASSES,
        "best_val_accuracy_pct": round(best_acc, 2),
        "dataset_sources": ["PlantDoc (Field)", "PlantVillage (Lab)"]
    }, f, indent=2)

print(f"\n🎉 Fine-Tuning Completed! Peak Validation Accuracy: {best_acc:.2f}%")


## 📦 Step 7: 1-Click Download Model to Your Local Computer
Run this cell to zip and download `model.safetensors` and `config.json` directly through your browser (~190 MB).


In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("agrihub_finetuned_detector", "zip", "./agrihub_finetuned_detector")
print(f"📦 Model weights archive created: {zip_path}")
print("⬇️ Initiating download to your computer...")
files.download(zip_path)


### 🌾 How to Deploy into AgriHub:
1. Extract `agrihub_finetuned_detector.zip` on your computer.
2. You will get two files:
   * `model.safetensors`
   * `config.json`
3. Place them into:
   `AgriHub/Crop recommendation/agrihub-ml/models/detectors/new/`
4. Now start AgriHub (`npm run dev`) — the detector will immediately recognize field-condition **Maka / Corn (Maize)** photos with high confidence!
